In [ ]:
import numpy as np
import optuna
from sklearn.metrics import average_precision_score
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler

from features import feature_engeneering, get_fe_parametrs
from folds import FOLDS, X, y


def objective_gmm(trial):
    params = {
        'n_components': trial.suggest_int('n_components', 1, 10),
        'covariance_type': trial.suggest_categorical('covariance_type', ['full', 'tied', 'diag']),
        'random_state': 131 
    }

    aucs = []
    for fold, (tr_idx, val_idx) in enumerate(FOLDS):
        X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

        X_tr_f = feature_engeneering(df=X_tr, qt=qt, train_median=train_median,
                                     train_mad=train_mad, tsh_quantile=tsh_q,
                                     fti_quantile=fti_q, tt4_quantile=tt4_q)
        X_val_f = feature_engeneering(df=X_val, qt=qt, train_median=train_median,
                                      train_mad=train_mad, tsh_quantile=tsh_q,
                                      fti_quantile=fti_q, tt4_quantile=tt4_q)


        X_tr_normal = X_tr_f[y_tr == 0]
        
        scaler = StandardScaler()
        scaler.fit(X_tr_normal)
        X_tr_scaled = scaler.transform(X_tr_normal)
        X_val_scaled = scaler.transform(X_val_f)

        gmm = GaussianMixture(**params)
        gmm.fit(X_tr_scaled)

        val_scores = -gmm.score_samples(X_val_scaled)

        pr_auc = average_precision_score(y_val, val_scores)
        aucs.append(pr_auc)

        trial.report(float(np.mean(aucs)), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(aucs))

study_gmm = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=131),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
)
study_gmm.optimize(objective_gmm, n_trials=100)

print(f'Best_params: {study_gmm.best_params}')
print(f'Best_value: {study_gmm.best_value}')


c:\Users\DNS\PycharmProjects\ds&ml\anomaly_and_pca\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-11-22 16:25:27,449] A new study created in memory with name: no-name-cdffec36-8cdc-415f-b1d6-d39a0790bc2f
[I 2026-11-22 16:25:31,645] Trial 0 finished with value: 0.6267461839916358 and parameters: {'n_components': 7, 'covariance_type': 'full'}. Best is trial 0 with value: 0.6267461839916358.
[I 2026-11-22 16:25:32,046] Trial 1 finished with value: 0.5670683629193098 and parameters: {'n_components': 7, 'covariance_type': 'diag'}. Best is trial 0 with value: 0.6267461839916358.
[I 2026-11-22 16:25:32,732] Trial 2 finished with value: 0.5969881013555017 and parameters: {'n_components': 5, 'covariance_type': 'full'}. Best is trial 0 with value: 0.6267461839916358.
[I 2026-11-22 16:25:33,111] Trial 3 finishe

Best_params: {'n_components': 7, 'covariance_type': 'full'}
Best_value: 0.6267461839916358


In [ ]:
best_params = study_gmm.best_params
oof_gmm = np.zeros(len(X))
synth_gmm = []

for fold, (tr_idx, val_idx) in enumerate(FOLDS):
    X_tr, y_tr = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    qt, train_median, train_mad, tsh_q, fti_q, tt4_q = get_fe_parametrs(X_tr)

    X_tr_f = feature_engeneering(
        df=X_tr, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q, 
        fti_quantile=fti_q, tt4_quantile=tt4_q 
    )
    X_val_f = feature_engeneering(
        df=X_val, qt=qt, train_median=train_median,
        train_mad=train_mad, tsh_quantile=tsh_q,
        fti_quantile=fti_q, tt4_quantile=tt4_q
    )

    X_tr_normal = X_tr_f[y_tr == 0]

    scaler = StandardScaler()
    scaler.fit(X_tr_normal)

    X_tr_scaled = scaler.transform(X_tr_normal)
    X_val_scaled = scaler.transform(X_val_f)

    X_synth = X_tr_normal.copy()
    for col in X_tr_normal.columns:
        X_synth[col] = np.random.permutation(X_synth[col].values)
    X_synth_scaled = scaler.transform(X_synth)


    model = GaussianMixture(**best_params)
    model.fit(X_tr_scaled)

    oof_gmm[val_idx] = -model.score_samples(X_val_scaled)
    synth_gmm.extend(-model.score_samples(X_synth_scaled))

np.save('oof_gmm.npy', oof_gmm)
np.save('synth_gmm.npy', synth_gmm)

: 